## BPE Tokenizers (SentencePiece)

### 1 Character Frequency and Coverage Analysis

In [2]:
# imports
from pathlib import Path
from collections import Counter
import sentencepiece as spm

DATA_DIR = Path("/srv/data/lt2326-h26/a1")

#### Objective

In [4]:
# Count character occurrences across all three training languages
char_counts = Counter()

for lang in ["en", "tr", "zh"]:
    file_path = DATA_DIR / "train" / f"{lang}.txt"

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            char_counts.update(line.rstrip("\r\n"))

# Calculate how many characters are needed for each coverage level
total_chars = sum(char_counts.values())

for coverage in [0.90, 0.95, 0.97, 0.99, 0.995, 0.999, 0.9995, 1.0]:
    cumulative = 0
    required_chars = 0

    for char, count in char_counts.most_common():
        cumulative += count
        required_chars += 1

        if cumulative / total_chars >= coverage:
            break

    print(f"Coverage {coverage:.4f}: {required_chars:,} characters")

Coverage 0.9000: 457 characters
Coverage 0.9500: 973 characters
Coverage 0.9700: 1,422 characters
Coverage 0.9900: 2,533 characters
Coverage 0.9950: 3,335 characters
Coverage 0.9990: 5,497 characters
Coverage 0.9995: 6,490 characters
Coverage 1.0000: 9,442 characters


### 2 Language Specific Character Coverage

In [5]:
# Select the most frequent characters covering 95% of the combined corpus

coverage_target = 0.95
cumulative = 0
selected_chars = set()

for char, count in char_counts.most_common():
    selected_chars.add(char)
    cumulative += count

    if cumulative / total_chars >= coverage_target:
        break

print(f"Characters selected: {len(selected_chars):,}")

# Measure how well these characters cover each language

for lang in ["en", "tr", "zh"]:

    file_path = DATA_DIR / "train" / f"{lang}.txt"

    lang_counts = Counter()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            lang_counts.update(line.rstrip("\r\n"))

    total = sum(lang_counts.values())

    covered = sum(
        count
        for char, count in lang_counts.items()
        if char in selected_chars
    )

    excluded = total - covered

    print(f"\nLanguage: {lang}")
    print(f"Character coverage: {covered / total:.2%}")
    print(f"Character occurrences excluded: {excluded:,}")
    print(f"Distinct characters excluded: {len(set(lang_counts) - selected_chars):,}")

Characters selected: 973

Language: en
Character coverage: 99.92%
Character occurrences excluded: 2,422
Distinct characters excluded: 493

Language: tr
Character coverage: 99.91%
Character occurrences excluded: 2,519
Distinct characters excluded: 377

Language: zh
Character coverage: 85.17%
Character occurrences excluded: 436,786
Distinct characters excluded: 8,245


#### Comparing Character Coverage Thresholds

In [10]:
# Compare language-specific coverage at different thresholds

coverage_levels = [0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999, 0.9995, 1.0]

# Count characters separately for each language
language_counts = {}

for lang in ["en", "tr", "zh"]:
    file_path = DATA_DIR / "train" / f"{lang}.txt"

    counts = Counter()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            counts.update(line.rstrip("\r\n"))

    language_counts[lang] = counts

# Calculate coverage for each language at each threshold
for target in coverage_levels:

    cumulative = 0
    selected = set()

    for char, count in char_counts.most_common():
        selected.add(char)
        cumulative += count

        if cumulative / total_chars >= target:
            break

    print(f"\nOverall coverage target: {target:.0%}")
    print(f"Characters selected: {len(selected):,}")

    for lang, counts in language_counts.items():

        total = sum(counts.values())

        covered = sum(
            count for char, count in counts.items()
            if char in selected
        )

        print(f"  {lang}: {covered / total:.2%}")


Overall coverage target: 90%
Characters selected: 457
  en: 99.86%
  tr: 99.73%
  zh: 70.44%

Overall coverage target: 95%
Characters selected: 973
  en: 99.92%
  tr: 99.91%
  zh: 85.17%

Overall coverage target: 97%
Characters selected: 1,422
  en: 99.93%
  tr: 99.93%
  zh: 91.14%

Overall coverage target: 98%
Characters selected: 1,805
  en: 99.94%
  tr: 99.95%
  zh: 94.11%

Overall coverage target: 99%
Characters selected: 2,533
  en: 99.96%
  tr: 99.96%
  zh: 97.08%

Overall coverage target: 100%
Characters selected: 3,335
  en: 99.97%
  tr: 99.97%
  zh: 98.56%

Overall coverage target: 100%
Characters selected: 5,497
  en: 99.99%
  tr: 99.99%
  zh: 99.72%

Overall coverage target: 100%
Characters selected: 6,490
  en: 100.00%
  tr: 100.00%
  zh: 99.86%

Overall coverage target: 100%
Characters selected: 9,442
  en: 100.00%
  tr: 100.00%
  zh: 100.00%


### 3 SentencePiece BPE Training

#### 3.1 Training Corpus Verification

In [11]:
# Locate the prepared multilingual training corpus
TOKENIZER_DATA = DATA_DIR / "tokenizer" / "balanced.txt"

print("File exists:", TOKENIZER_DATA.exists())

if TOKENIZER_DATA.exists():
    print(f"File size: {TOKENIZER_DATA.stat().st_size / 1_000_000:.2f} MB")

    print("\nFirst five lines:")

    with open(TOKENIZER_DATA, "r", encoding="utf-8") as f:
        for _ in range(5):
            print(repr(f.readline()[:200]))

File exists: True
File size: 14.24 MB

First five lines:
'But French people use the word county (comté) in the name of the Free County region, the old Free County of Burgundy.\n'
'Daha sonra buna ek olarak Angel yayın yapmaya başlamıştır.\n'
'Formül aynı zamanda gözlemlenemeyen morötesi ve kızılötesi ışıkların bazı ek spektral çizgilerini tahmin etmiştir.\n'
"İGD Dış Bürosu tarafından 1980'li yıllarda yayınlanan haber bülteni Türkçenin yanı sıra İngilizce ve Fransızca olarak da çıkmıştır.\n"
'Bu durum sayesinde düşmanlarından gizlenir.\n'


#### 3.2 Model Output Directory

In [12]:
# Directory for saved tokenizer models
TOKENIZER_DIR = Path("../models/tokenizers")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

print("Tokenizer output directory:", TOKENIZER_DIR.resolve())

Tokenizer output directory: /home/gussougr@GU.GU.SE/lt2326-assignment1/models/tokenizers
